# Feature engineering

Primero haremos modelos para cada familia de productos y ver que tal funciona

In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [106]:
TRAIN_CSV_PATH = '../data/raw/store-sales-time-series-forecasting/train.csv'

In [107]:
train = pd.read_csv(TRAIN_CSV_PATH, parse_dates=['date'])
train['date'] = pd.to_datetime(train['date'])

In [108]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [109]:
grouped_train = train.groupby(['family', 'date']).agg({'sales': 'sum', 'onpromotion': 'sum'}).reset_index()

grouped_train.head()

,family,date,sales,onpromotion
0,AUTOMOTIVE,2013-01-01,0.0,0
1,AUTOMOTIVE,2013-01-02,255.0,0
2,AUTOMOTIVE,2013-01-03,161.0,0
3,AUTOMOTIVE,2013-01-04,169.0,0
4,AUTOMOTIVE,2013-01-05,342.0,0


In [110]:
families = grouped_train['family'].value_counts().index

In [111]:
dataframes = {}


for family in families:
    family_df = grouped_train[grouped_train['family'] == family].copy()
    family_df.set_index('date', inplace=True)
    family_df = family_df.asfreq('D')
    dataframes[family] = family_df
    

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion
date,,,
2013-01-01,AUTOMOTIVE,0.0,0.0
2013-01-02,AUTOMOTIVE,255.0,0.0
2013-01-03,AUTOMOTIVE,161.0,0.0
2013-01-04,AUTOMOTIVE,169.0,0.0
2013-01-05,AUTOMOTIVE,342.0,0.0


In [112]:
#Ver si hay valores nulos en los dataframes
for family, df in dataframes.items():
    if df.isnull().values.any():
        print(f"Missing values found in {family} dataframe.")
    else:
        print(f"No missing values in {family} dataframe.")

Missing values found in AUTOMOTIVE dataframe.
Missing values found in BABY CARE dataframe.
Missing values found in BEAUTY dataframe.
Missing values found in BEVERAGES dataframe.
Missing values found in BOOKS dataframe.
Missing values found in BREAD/BAKERY dataframe.
Missing values found in CELEBRATION dataframe.
Missing values found in CLEANING dataframe.
Missing values found in DAIRY dataframe.
Missing values found in DELI dataframe.
Missing values found in EGGS dataframe.
Missing values found in FROZEN FOODS dataframe.
Missing values found in GROCERY I dataframe.
Missing values found in GROCERY II dataframe.
Missing values found in HARDWARE dataframe.
Missing values found in HOME AND KITCHEN I dataframe.
Missing values found in HOME AND KITCHEN II dataframe.
Missing values found in HOME APPLIANCES dataframe.
Missing values found in HOME CARE dataframe.
Missing values found in LADIESWEAR dataframe.
Missing values found in LAWN AND GARDEN dataframe.
Missing values found in LINGERIE dat

In [113]:
#Rellenamos family con cada familia, sales con la media y onpromotion con 0
for family, df in dataframes.items():
    df['family'] = family
    df['sales'] = df['sales'].fillna(df['sales'].mean())
    df['onpromotion'] = df['onpromotion'].fillna(0)

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion
date,,,
2013-01-01,AUTOMOTIVE,0.0,0.0
2013-01-02,AUTOMOTIVE,255.0,0.0
2013-01-03,AUTOMOTIVE,161.0,0.0
2013-01-04,AUTOMOTIVE,169.0,0.0
2013-01-05,AUTOMOTIVE,342.0,0.0


In [114]:
# Restamos la fecha a la fecha mínima para obtener el número de días desde el inicio

for family, df in dataframes.items():
    df['days_since_start'] = (df.index - df.index.min()).days + 1


dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion,days_since_start
date,,,,
2013-01-01,AUTOMOTIVE,0.0,0.0,1
2013-01-02,AUTOMOTIVE,255.0,0.0,2
2013-01-03,AUTOMOTIVE,161.0,0.0,3
2013-01-04,AUTOMOTIVE,169.0,0.0,4
2013-01-05,AUTOMOTIVE,342.0,0.0,5


In [115]:
dataframes_targets['AUTOMOTIVE'].head()

date
2013-01-01      0.0
2013-01-02    255.0
2013-01-03    161.0
2013-01-04    169.0
2013-01-05    342.0
Freq: D, Name: sales, dtype: float64

In [116]:
# Quitamos la columna family y onpromotion de los features ya que no es necesaria
#Guardamos primero la columna onpromotion en un diccionario para poder usarla en el modelo de XGBoost
onpromotion = {}


for family, df in dataframes.items():
    onpromotion[family] = df['onpromotion'].copy()
    df.drop(columns=['family', 'onpromotion'], inplace=True)

In [117]:
# Los partimos en train y test, dejando los últimos 90 días para test
for family, df in dataframes.items():
    train_df = df.iloc[:-90]
    test_df = df.iloc[-90:]
    dataframes[family] = (train_df, test_df)

Ahora empezamos con los datos para XGBoost  
No restaremos a los 'sales' la tendencia, lo haremos en el siguiente notebook

In [118]:
# Pondremos features de dia y mes que es
for family, (train_df, test_df) in dataframes.items():


    train_df['Month'] = train_df.index.month
    train_df['Day'] = train_df.index.day

    test_df['Month'] = test_df.index.month
    test_df['Day'] = test_df.index.day

    dataframes[family] = (train_df, test_df)

C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['Month'] = train_df.index.month
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['Day'] = train_df.index.day
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [119]:
#Teniendo todo alineamos la columna onpromotion con los dataframes de train y test para poder usarla en el modelo de XGBoost
for family, (train_df, test_df) in dataframes.items():

    train_onpromotion = onpromotion[family].iloc[:-90]
    test_onpromotion = onpromotion[family].iloc[-90:]

    train_df['onpromotion'] = train_onpromotion.values
    test_df['onpromotion'] = test_onpromotion.values

    dataframes[family] = (train_df, test_df)

C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2757716268.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['onpromotion'] = train_onpromotion.values
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2757716268.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['onpromotion'] = test_onpromotion.values


In [120]:
dataframes['AUTOMOTIVE'][0].head()

,sales,days_since_start,Month,Day,onpromotion
date,,,,,
2013-01-01,0.0,1,1,1,0.0
2013-01-02,255.0,2,1,2,0.0
2013-01-03,161.0,3,1,3,0.0
2013-01-04,169.0,4,1,4,0.0
2013-01-05,342.0,5,1,5,0.0


In [121]:
# Lo guardamos los dos conjuntos de dataframes en distintos archivos csv para poder usarlos en el siguiente notebook
os.makedirs('../data/processed/LinearandXGBoost/train', exist_ok=True)
os.makedirs('../data/processed/LinearandXGBoost/test', exist_ok=True)

for family, (train_df, test_df) in dataframes.items():
    family = family.replace("/", "_")  # Reemplaza las barras por guiones bajos en el nombre de la familia
    train_df.to_csv(f'../data/processed/LinearandXGBoost/train/{family}_train.csv', index=True)
    test_df.to_csv(f'../data/processed/LinearandXGBoost/test/{family}_test.csv', index=True)